# FraudTrace AI - 02: Forensic Data Cleaning & Robust Normalization

> **DISCLAIMER: SYNTHETIC DATA ONLY**
> All data used in this notebook is synthetically generated for research, testing, and demonstration purposes. No real police records, personal identification, or banking transactions are included. This system does not make legal guilt determinations; it reconstructs and correlates evidentiary relationships.

### Purpose
Demonstrate normalization across heterogeneous evidence formats (phone numbers, amounts, UPI IDs, timestamps).

| Attribute | Description |
| :--- | :--- |
| **Input** | Raw heterogeneous evidence items with dirty formats |
| **Output** | Cleaned, standardized evidence objects saved to ml/data/processed/ |
| **Target Environment** | Google Colab (CPU / GPU compatible) & Local |
| **Package** | `ml.src` |


In [ ]:
# ====================================================================
# GOOGLE COLAB ENVIRONMENT SETUP
# ====================================================================
import os
import sys

# Clone repository if running inside Google Colab
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('FraudTrace.AI'):
        !git clone https://github.com/Cyberdude441/FraudTrace.AI.git
    %cd FraudTrace.AI
    !pip install -r ml/requirements-colab.txt

# Ensure repository root is on sys.path for local and Colab execution
repo_root = os.path.abspath('.')
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"[OK] Environment initialized. Working directory: {os.getcwd()}")


## 1. Phone & Identifier Normalization
Real-world evidence logs phones in varying formats: `+91 98765-43210`, `09876543210`, `98765 43210`. We normalize all Indian numbers into canonical 10-digit strings.

In [ ]:
from ml.src.preprocessing.entity_normalizer import EntityNormalizer

raw_phones = [
    "+91 98765-43210",
    "09876543210",
    "9876543210",
    "+91-9876543210",
    "98765 43210"
]

print("=== Phone Normalization ===")
for p in raw_phones:
    clean = EntityNormalizer.normalize_phone(p)
    print(f"Raw: {p:<18} -> Normalized: {clean}")
    assert clean == "9876543210"
print("[OK] All phone variants normalized to canonical format.")


## 2. Currency Amount Parsing
We parse amounts formatted with Rupee symbols, commas, decimals, and words.

In [ ]:
from ml.src.preprocessing.amount_normalizer import AmountNormalizer

raw_amounts = [
    "₹50,000.00",
    "Rs. 50000",
    "INR 48,000.50",
    50000,
    "48000"
]

print("=== Amount Normalization ===")
for a in raw_amounts:
    clean = AmountNormalizer.parse_amount(a)
    print(f"Raw: {str(a):<18} -> Parsed Float: {clean}")


## 3. Timestamp Normalization & Drift Detection
Forensic sources report timestamps in UTC, local IST, or varied formats. We parse to ISO-8601 UTC and compute drift.

In [ ]:
from ml.src.preprocessing.timestamp_normalizer import TimestampNormalizer

t1 = "2024-03-15T10:30:00Z"
t2 = "2024-03-15T12:30:00Z"

drift_sec = TimestampNormalizer.compute_drift_seconds(t1, t2)
drift_min = TimestampNormalizer.compute_drift_minutes(t1, t2)
print(f"Timestamp 1 : {t1}")
print(f"Timestamp 2 : {t2}")
print(f"Drift (Sec) : {drift_sec} seconds")
print(f"Drift (Min) : {drift_min} minutes (2 hours drift)")


## 4. Run Dataset Cleaner & Save to `ml/data/processed/`
Clean all sample evidence records and persist the cleaned dataset.

In [ ]:
from ml.src.data.cleaner import DataCleaner
from ml.src.data.loader import DataLoader
from ml.src.config import get_data_dir

loader = DataLoader(dataset_type="sample")
raw_evidence = loader.load_evidence()

cleaner = DataCleaner()
cleaned_evidence = [cleaner.clean_evidence_item(e) for e in raw_evidence]

processed_dir = get_data_dir("processed")
processed_dir.mkdir(parents=True, exist_ok=True)
out_file = processed_dir / "cleaned_evidence.json"

import json
with open(out_file, "w", encoding="utf-8") as f:
    json.dump(cleaned_evidence, f, indent=2)

print(f"[OK] Successfully cleaned {len(cleaned_evidence)} evidence items and saved to {out_file}")
